# 01 — Data and features
Inspect the real input format, user support and the two streams used by submission B. This notebook does not fit models. See [the method](../docs/METHOD.md) for the complete recipe.

In [ ]:
import os
from pathlib import Path
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[name] = '1'
import pandas as pd
from copilote_b.data import data_path, load_sessions, load_sample
from copilote_b.features import prepare
from copilote_b.model import crop_plan
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/copilote_b').is_dir())
default_data = PROJECT.parent / 'data' if (PROJECT.parent / 'data/train.csv').exists() else PROJECT / 'data'
DATA = Path(os.environ.get('COPILOTE_DATA', default_data)).resolve()
print(DATA)

## Browser-first test parsing
Training starts with a trigram; test starts with the browser. Records are variable-length and may contain protected commas. We validate the sample before creating predictions.

In [ ]:
train = load_sessions(data_path(DATA, 'train.csv'), labeled=True)
test = load_sessions(data_path(DATA, 'test.csv'), labeled=False)
sample = load_sample(DATA, len(test))
print({'training_rows': len(train), 'test_rows': len(test), 'sample_shape': sample.shape})
print('First training label/browser:', train[0].label, train[0].browser)
print('First test label/browser:', test[0].label, test[0].browser)
assert all(s.label is None for s in test)
display(sample.head())

In [ ]:
sessions = pd.DataFrame([{'user': s.label, 'browser': s.browser, 'actions': len(s.actions), 'windows': len(s.markers), 'duration_bound': max(s.markers, default=0)} for s in train])
support = sessions.user.value_counts().rename('sessions')
display(support.describe().to_frame())
display(sessions[['actions', 'windows', 'duration_bound']].describe(percentiles=[.25, .5, .75, .95]))
display(pd.crosstab(sessions.user, sessions.browser).head(12))
print('Users with multiple observed browsers:', int((sessions.groupby('user').browser.nunique() > 1).sum()))

## Aligned action and context streams
The feature builder never receives a user label. Explicit time markers are excluded from these final models. Screen, configuration and chain context are carried forward; a new screen resets the latter two when absent.

In [ ]:
trace = prepare(train[0])
assert len(trace.full) == len(trace.context) == len(train[0].actions)
display(pd.DataFrame({'full_action': trace.full[:8], 'carried_context': trace.context[:8]}))
print('Original action count:', len(trace.full))
print('128-action training plan:', crop_plan(len(trace.full), 128))
print('Worked 340-action example:', crop_plan(340, 128))

## Next
Continue with [training and validation](02_training_and_validation.ipynb). The last time marker is a duration bound, not an exact per-action timestamp. Browser support is learned inside each fitting partition, not from the whole labeled dataset during CV.